# HEST-1k HisToGene Patch-H5 Adapter Smoke Test

This notebook verifies that the coverage95 HEST patch-H5 adapter exposes HisToGene-style inputs without using WSI crops: RGB patch tensor, normalized position, log1p_rate expression target, and measured-gene mask.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np

from histoomnist.data.gene_selection import gene_key_settings_from_config
from histoomnist.eval.benchmark_predictions import load_slide_target
from histoomnist.external.histogene_patch_h5 import HistogenePatchH5Dataset, target_values_from_counts
from histoomnist.utils.config import load_config
from histoomnist.utils.io import read_manifest
from histoomnist.utils.project_paths import resolve_project_path

In [2]:
cfg = load_config(resolve_project_path("configs/hest1k_human_visium_expression_highconf_symbol95.yaml"))
dataset = HistogenePatchH5Dataset(cfg, splits=["test"], max_slides=2, target_kind="log1p_rate")
slide_summary = dataset.slide_summary_frame().drop(columns=["patch_h5_path"])
{
    "n_slides": len(dataset.slides),
    "n_spots": len(dataset),
    "n_target_genes": len(dataset.target_genes),
    "slides": slide_summary.to_dict(orient="records"),
}

{'n_slides': 2,
 'n_spots': 7893,
 'n_target_genes': 16942,
 'slides': [{'sample_id': 'MISC33',
   'split': 'test',
   'organ': 'Bowel',
   'cohort': 'COLON MAP: Colon Molecular Atlas Project',
   'n_spots': 3385,
   'n_measured_target_genes': 16426},
  {'sample_id': 'MISC46',
   'split': 'test',
   'organ': 'Bowel',
   'cohort': 'COLON MAP: Colon Molecular Atlas Project',
   'n_spots': 4508,
   'n_measured_target_genes': 16924}]}

In [3]:
checked = []
for idx in range(8):
    item = dataset[idx]
    patch = item["patch"].numpy()
    target = item["log1p_rate"].numpy()
    mask = item["expression_mask"].numpy().astype(bool)
    checked.append(
        {
            "index": idx,
            "sample_id": item["sample_id"],
            "patch_shape": tuple(patch.shape),
            "patch_min": float(patch.min()),
            "patch_max": float(patch.max()),
            "target_dim": int(target.shape[0]),
            "measured_genes": int(mask.sum()),
            "target_finite_fraction": float(np.isfinite(target[mask]).mean()),
        }
    )
assert all(row["patch_shape"] == (3, 224, 224) for row in checked)
assert min(row["patch_min"] for row in checked) >= 0.0
assert max(row["patch_max"] for row in checked) <= 1.0
assert all(row["target_dim"] == len(dataset.target_genes) for row in checked)
assert all(row["target_finite_fraction"] == 1.0 for row in checked)
checked[:3]

[{'index': 0,
  'sample_id': 'MISC33',
  'patch_shape': (3, 224, 224),
  'patch_min': 0.0,
  'patch_max': 1.0,
  'target_dim': 16942,
  'measured_genes': 16426,
  'target_finite_fraction': 1.0},
 {'index': 1,
  'sample_id': 'MISC33',
  'patch_shape': (3, 224, 224),
  'patch_min': 0.0,
  'patch_max': 1.0,
  'target_dim': 16942,
  'measured_genes': 16426,
  'target_finite_fraction': 1.0},
 {'index': 2,
  'sample_id': 'MISC33',
  'patch_shape': (3, 224, 224),
  'patch_min': 0.0,
  'patch_max': 1.0,
  'target_dim': 16942,
  'measured_genes': 16426,
  'target_finite_fraction': 1.0}]

In [4]:
manifest_path = resolve_project_path(cfg["data"]["manifest"])
manifest = read_manifest(manifest_path)
first_slide = dataset.slides[0]
row = manifest[manifest["sample_id"].astype(str).eq(first_slide.sample_id)].iloc[0]
gene_key, raw_st_root = gene_key_settings_from_config(cfg)
raw_st_root = resolve_project_path(raw_st_root) if raw_st_root is not None else None
benchmark_target = load_slide_target(
    row=row,
    base_dir=manifest_path.parent,
    target_genes=dataset.target_genes,
    gene_key=gene_key,
    raw_st_root=raw_st_root,
    min_total_counts=float(cfg["data"].get("min_total_counts", 1.0)),
)
max_abs_diff = 0.0
for local_idx in range(8):
    adapter = target_values_from_counts(
        first_slide.counts.getrow(local_idx).toarray().reshape(-1),
        float(first_slide.size_factor[local_idx]),
        "log1p_rate",
    )
    benchmark = target_values_from_counts(
        benchmark_target.counts.getrow(local_idx).toarray().reshape(-1),
        float(benchmark_target.size_factor[local_idx]),
        "log1p_rate",
    )
    max_abs_diff = max(max_abs_diff, float(np.max(np.abs(adapter - benchmark))))
consistency = {
    "sample_id": first_slide.sample_id,
    "checked_rows": 8,
    "spot_ids_match": first_slide.spot_ids[:8] == benchmark_target.spot_ids[:8],
    "measured_gene_mask_match": bool(np.array_equal(first_slide.measured_genes, benchmark_target.measured_genes)),
    "max_abs_target_diff": max_abs_diff,
}
assert consistency["spot_ids_match"]
assert consistency["measured_gene_mask_match"]
assert consistency["max_abs_target_diff"] == 0.0
consistency

{'sample_id': 'MISC33',
 'checked_rows': 8,
 'spot_ids_match': True,
 'measured_gene_mask_match': True,
 'max_abs_target_diff': 0.0}